In [175]:
import pandas as pd
import numpy as np

In [176]:
url = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv'
df = pd.read_csv(url)
df.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [177]:
df.describe()

,model_year,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
count,10000.000000,10000.00000,10000.000000,10000.000000,9123.000000,10000.000000,9736.000000,10000.000000
mean,1999.549500,3.50400,2268.807000,6.035200,254.446016,4286.754000,18.888044,29.997700
std,14.207925,0.95209,183.447355,0.542578,22.844613,266.212491,1.279543,2.930245
min,1975.000000,2.00000,1590.000000,4.000000,171.000000,3320.000000,14.000000,19.800000
25%,1987.000000,3.00000,2150.000000,6.000000,239.000000,4110.000000,18.000000,28.000000
50%,2000.000000,4.00000,2270.000000,6.000000,254.000000,4280.000000,18.900000,30.000000
75%,2012.000000,4.00000,2390.000000,6.000000,270.000000,4470.000000,19.700000,31.900000
max,2024.000000,5.00000,3110.000000,8.000000,334.000000,5460.000000,23.800000,41.200000


In [178]:
df.isnull().sum()

,0
model_year,0
origin,0
fuel_type,0
drivetrain,0
num_doors,0
engine_displacement,0
num_cylinders,0
horsepower,877
vehicle_weight,0
acceleration,264


In [179]:
df = df.fillna(0)

In [180]:
df.isnull().sum()

,0
model_year,0
origin,0
fuel_type,0
drivetrain,0
num_doors,0
engine_displacement,0
num_cylinders,0
horsepower,0
vehicle_weight,0
acceleration,0


In [181]:
from sklearn.model_selection import train_test_split

In [182]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

In [183]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [184]:
y_train = df_train.fuel_efficiency_mpg.values
y_val = df_val.fuel_efficiency_mpg.values
y_test = df_test.fuel_efficiency_mpg.values

In [185]:
del df_train['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']
del df_test['fuel_efficiency_mpg']

In [186]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import roc_auc_score
from sklearn.tree import export_text

In [187]:
dv = DictVectorizer(sparse=True)

In [188]:
train_dicts = df_train.to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

Q1

In [189]:
dtr = DecisionTreeRegressor(max_depth=1, random_state=1)
dtr.fit(X_train, y_train)

DecisionTreeRegressor(max_depth=1, random_state=1)

In [190]:
print(export_text(dtr, feature_names=list(dv.get_feature_names_out())))

|--- model_year <= 1997.50
|   |--- value: [28.61]
|--- model_year >  1997.50
|   |--- value: [31.17]



Q2

In [191]:
from sklearn.ensemble import RandomForestRegressor

In [192]:
rfr = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
rfr.fit(X_train, y_train)

RandomForestRegressor(n_estimators=10, n_jobs=-1, random_state=1)

In [193]:
from sklearn.metrics import root_mean_squared_error

In [194]:
val_dicts = df_val.to_dict(orient='records')
X_val = dv.transform(val_dicts)

y_pred = rfr.predict(X_val)

In [195]:
rmse = root_mean_squared_error(y_val, y_pred)
round(rmse, 3)

1.837

Q3

In [196]:
for n in [10, 50, 100, 150]:
  rfr = RandomForestRegressor(n_estimators=n, random_state=1, n_jobs=-1)
  rfr.fit(X_train, y_train)

  y_pred = rfr.predict(X_val)

  rmse = root_mean_squared_error(y_val, y_pred)
  print(n, '=', round(rmse, 3))

10 = 1.837
50 = 1.772
100 = 1.768
150 = 1.769


Q4

In [197]:
for d in [10, 15, 20, 25]:
  for n in [10, 50, 100, 150]:
    rfr = RandomForestRegressor(n_estimators=n, max_depth=d, random_state=1, n_jobs=-1)
    rfr.fit(X_train, y_train)

    y_pred = rfr.predict(X_val)

    rmse = root_mean_squared_error(y_val, y_pred)
    print(f'd={d}, n={n}, rmse={round(rmse, 3)}')

d=10, n=10, rmse=1.784
d=10, n=50, rmse=1.749
d=10, n=100, rmse=1.745
d=10, n=150, rmse=1.745
d=15, n=10, rmse=1.83
d=15, n=50, rmse=1.771
d=15, n=100, rmse=1.768
d=15, n=150, rmse=1.767
d=20, n=10, rmse=1.841
d=20, n=50, rmse=1.773
d=20, n=100, rmse=1.769
d=20, n=150, rmse=1.77
d=25, n=10, rmse=1.837
d=25, n=50, rmse=1.772
d=25, n=100, rmse=1.768
d=25, n=150, rmse=1.768


Q5

In [198]:
rfr = RandomForestRegressor(n_estimators=10, max_depth=20, random_state=1, n_jobs=-1)
rfr.fit(X_train, y_train)

RandomForestRegressor(max_depth=20, n_estimators=10, n_jobs=-1, random_state=1)

In [199]:
imp = pd.Series(rfr.feature_importances_, index=dv.get_feature_names_out()).sort_values(ascending=False)
print(imp)

model_year                      0.296001
vehicle_weight                  0.203879
fuel_type=Gasoline              0.160712
horsepower                      0.078017
engine_displacement             0.061005
acceleration                    0.055201
origin=USA                      0.042144
drivetrain=Front-wheel drive    0.020145
num_doors                       0.019415
fuel_type=Diesel                0.017038
fuel_type=Hybrid                0.015800
drivetrain=All-wheel drive      0.007682
num_cylinders                   0.007284
origin=Asia                     0.006391
drivetrain=Rear-wheel drive     0.005781
origin=Europe                   0.003505
dtype: float64


Q6

In [206]:
import xgboost as xgb

In [207]:
features = list(dv.get_feature_names_out())
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=features)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=features)

In [208]:
watchlist = [(dtrain, 'train'), (dval, 'val')]

In [214]:
xgb_params = {
    'eta': 0.1,
    'max_depth': 6,
    'min_child_weight': 1,

    'objective': 'reg:squarederror',
    'nthread': 8,

    'seed': 1,
    'verbosity': 1,
}

model = xgb.train(xgb_params, dtrain, num_boost_round=100,
                  verbose_eval=5,
                  evals=watchlist)

[0]	train-rmse:2.74739	val-rmse:2.78635
[5]	train-rmse:2.12109	val-rmse:2.20428
[10]	train-rmse:1.80592	val-rmse:1.93152
[15]	train-rmse:1.64286	val-rmse:1.80346
[20]	train-rmse:1.55847	val-rmse:1.75076
[25]	train-rmse:1.50918	val-rmse:1.72751
[30]	train-rmse:1.47544	val-rmse:1.71667
[35]	train-rmse:1.44978	val-rmse:1.71287
[40]	train-rmse:1.43074	val-rmse:1.71289
[45]	train-rmse:1.41543	val-rmse:1.71310
[50]	train-rmse:1.40368	val-rmse:1.71358
[55]	train-rmse:1.39009	val-rmse:1.71570
[60]	train-rmse:1.37884	val-rmse:1.71688
[65]	train-rmse:1.37172	val-rmse:1.71812
[70]	train-rmse:1.36283	val-rmse:1.71919
[75]	train-rmse:1.35099	val-rmse:1.72057
[80]	train-rmse:1.34578	val-rmse:1.72100
[85]	train-rmse:1.33480	val-rmse:1.72192
[90]	train-rmse:1.32546	val-rmse:1.72303
[95]	train-rmse:1.32292	val-rmse:1.72324
[99]	train-rmse:1.31192	val-rmse:1.72481


In [215]:
y_pred = model.predict(dval)
rmse = root_mean_squared_error(y_val, y_pred)
rmse

1.724810059827976